# Config

In [ ]:
# config.py
import torch

SEED = 42
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

N_MC = 3000
N_POINTS = 101
N_CHANNELS = 12
N_OUTPUTS = 11

N_CURVE = N_CHANNELS * N_POINTS

save_dir = "mosfet_surrogate_3000_10_10_25C"

param_names = [
    "vth0", "k1", "k2", "nfactor", "u0", "ua", "ub", "a0", "ags", "uc", "keta"
]

channel_names = [
    "logId_Vds_0p1_Vbs_0", 
    "logId_Vds_0p1_Vbs_2", 
    "logId_Vds_0p1_Vbs_6",
    "logId_Vds_0p1_Vbs_10",

    "Id_Vds_0p1_Vbs_0", 
    "Id_Vds_0p1_Vbs_2", 
    "Id_Vds_0p1_Vbs_6",
    "Id_Vds_0p1_Vbs_10",
    "Id_Vds_5p0_Vbs_0", 
    "Id_Vds_5p0_Vbs_2",

    "dId_dVg_Vds_0p1_Vbs_0",

    "Vg"
]

Y_LOG_COLS = [4, 5, 6, 9]     # u0, ua, ub, uc  -> log10(abs)
Y_NEG_LOG_COLS = []                
EPS = 1e-30

# Model

In [ ]:
# model_surrogate.py
from torch import nn

N_CURVE = N_CHANNELS * N_POINTS

class MosfetSurrogate(nn.Module):
    def __init__(self, n_params=N_OUTPUTS, n_curve=N_CURVE):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_params, 512),  nn.BatchNorm1d(512),  nn.ReLU(),
            nn.Linear(512, 1024),      nn.BatchNorm1d(1024), nn.ReLU(),
            nn.Linear(1024, 2048),     nn.BatchNorm1d(2048), nn.ReLU(),
            nn.Linear(2048, 2048),     nn.BatchNorm1d(2048), nn.ReLU(),
            nn.Linear(2048, n_curve),
        )

    def forward(self, p):              
        return self.net(p)             

# Dataprep

In [ ]:
# data_prep_surrogate.py
import os, joblib, numpy as np, pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

os.makedirs(save_dir, exist_ok=True)
np.random.seed(SEED)

# ---- chargement X (.npz) ----
data = np.load("dataset_3000/mosfet_X_dataset_3000.npz")
X = data["X"].astype(np.float32)
print("X chargé, shape =", X.shape)
assert X.shape == (N_MC, N_POINTS, N_CHANNELS), \
    f"X devrait être {(N_MC, N_POINTS, N_CHANNELS)}, reçu {X.shape}"

# ---- chargement Y + transformations log [7] ----
Y = pd.read_csv(
    "dataset_3000/Y_3000.csv",
    header=None,
    sep=";"
).values.astype(np.float32)
Yp = Y.copy()
Yp[:, Y_LOG_COLS] = np.log10(np.abs(Yp[:, Y_LOG_COLS]) + EPS)
Yp[:, Y_NEG_LOG_COLS] = np.log10(np.abs(Yp[:, Y_NEG_LOG_COLS]) + EPS)

# ---- split ----
X_tr, X_tmp, Y_tr, Y_tmp = train_test_split(X, Yp, test_size=0.20, random_state=SEED, shuffle=True)
X_val, X_te, Y_val, Y_te = train_test_split(X_tmp, Y_tmp, test_size=0.50, random_state=SEED, shuffle=True)

# ---- scaler X par canal ----
scaler_x = MinMaxScaler((0, 1))
def norm_x(A, fit=False):
    A2 = A.reshape(-1, N_CHANNELS)
    A2 = scaler_x.fit_transform(A2) if fit else scaler_x.transform(A2)
    return A2.reshape(A.shape).astype(np.float32)

X_tr_n  = norm_x(X_tr, fit=True)
X_val_n = norm_x(X_val)
X_te_n  = norm_x(X_te)

# ---- scaler Y ----
scaler_y = MinMaxScaler((0, 1))
Y_tr_n  = scaler_y.fit_transform(Y_tr).astype(np.float32)
Y_val_n = scaler_y.transform(Y_val).astype(np.float32)
Y_te_n  = scaler_y.transform(Y_te).astype(np.float32)

# ---- bornes physiques (espace transformé/log, ordre param_names) ----
lower = np.array([0.55, 0.60, 0.03, 0.60, -1.40, -9.35, 
                  -19.00, 0.40, -0.10, -11.70, -0.02],
                 dtype=np.float32)
upper = np.array([0.75, 0.80, 0.08, 1.40, -1.20, -8.95, 
                  -17.80, 1.20, 0.10, -10.30, 0.02],
                 dtype=np.float32)

# Normalisation des bornes avec le MÊME scaler_y (ordre param_names [12])
lower_norm = scaler_y.transform(lower.reshape(1, -1)).astype(np.float32)[0]
upper_norm = scaler_y.transform(upper.reshape(1, -1)).astype(np.float32)[0]

np.save(f"{save_dir}/param_bounds_lower_norm.npy", lower_norm)
np.save(f"{save_dir}/param_bounds_upper_norm.npy", upper_norm)
print("Bornes normalisées sauvegardées.")
print("lower_norm:", lower_norm)
print("upper_norm:", upper_norm)

# ---- format APLATI (N, 2222) : aplatissement (L, C) -> (C*L) cohérent ----
def to_conv(A):  return np.transpose(A, (0, 2, 1)).copy()          
def to_flat(A):  return to_conv(A).reshape(A.shape[0], -1).copy()  

# Format Conv (N, C, L) pour extracteur multi-têtes [8]
np.save(f"{save_dir}/X_train_conv.npy", to_conv(X_tr_n))
np.save(f"{save_dir}/X_val_conv.npy",   to_conv(X_val_n))
np.save(f"{save_dir}/X_test_conv.npy",  to_conv(X_te_n))

# Format aplati (N, 2222) pour surrogate (sortie) et extracteur MLP (entrée)
np.save(f"{save_dir}/X_train_flat.npy", to_flat(X_tr_n))
np.save(f"{save_dir}/X_val_flat.npy",   to_flat(X_val_n))
np.save(f"{save_dir}/X_test_flat.npy",  to_flat(X_te_n))

# Cibles params
np.save(f"{save_dir}/Y_train_norm.npy", Y_tr_n)
np.save(f"{save_dir}/Y_val_norm.npy",   Y_val_n)
np.save(f"{save_dir}/Y_test_norm.npy",  Y_te_n)
np.save(f"{save_dir}/Y_test_processed.npy", Y_te)

joblib.dump(scaler_x, f"{save_dir}/scaler_x_minmax.pkl")
joblib.dump(scaler_y, f"{save_dir}/scaler_y_minmax.pkl")
print("Préparation terminée.")
print("  X_conv :", to_conv(X_tr_n).shape, " X_flat :", to_flat(X_tr_n).shape)
print("  Y      :", Y_tr_n.shape)

# Train

In [ ]:
# train_surrogate.py
import os, numpy as np, torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader

np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Device:", device, "| CUDA:", torch.cuda.is_available())

batch_size, num_epochs, patience, lr = 64, 250, 50, 3e-5
best_model_path = os.path.join(save_dir, "surrogate_best.pth")

# ---- données : entrée = params (Y_norm), sortie = courbes (X_flat) ----
P_tr  = torch.tensor(np.load(f"{save_dir}/Y_train_norm.npy"), dtype=torch.float32)
P_val = torch.tensor(np.load(f"{save_dir}/Y_val_norm.npy"),   dtype=torch.float32)
C_tr  = torch.tensor(np.load(f"{save_dir}/X_train_flat.npy"), dtype=torch.float32)
C_val = torch.tensor(np.load(f"{save_dir}/X_val_flat.npy"),   dtype=torch.float32)
print("Params:", P_tr.shape, "Courbes:", C_tr.shape)

train_loader = DataLoader(TensorDataset(P_tr, C_tr), batch_size=batch_size,
                          shuffle=True, drop_last=True,
                          pin_memory=torch.cuda.is_available())
val_loader = DataLoader(TensorDataset(P_val, C_val), batch_size=batch_size,
                        shuffle=False, pin_memory=torch.cuda.is_available())

model = MosfetSurrogate().to(device)
print(model)
print("Params entraînables :", sum(p.numel() for p in model.parameters() if p.requires_grad))

# ============================================================
# POIDS PAR CANAL (taille 22, étendus sur les 101 points -> 2222)
# ============================================================

channel_weights = torch.ones(N_CHANNELS, device=device)
for ch in [1, 2, 3, 5, 6, 7, 9, 10]:  
    channel_weights[ch] = 2.0

weight_vec = channel_weights.repeat_interleave(N_POINTS)   

print("\nPoids par canal appliqués :")
for i, name in enumerate(channel_names):
    print(f"  [{i:2d}] {name:>22s} : {channel_weights[i].item():.1f}")

# ---- loss Huber pondérée par canal (réduction manuelle) ----
huber = nn.HuberLoss(delta=0.5, reduction="none")

def weighted_curve_loss(pred, target):
    per_elem = huber(pred, target)         
    per_elem = per_elem * weight_vec          
    return per_elem.mean()

optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, "min",
                                                       factor=0.5, patience=10, min_lr=1e-7)

best_val, patience_counter = np.inf, 0
for epoch in range(num_epochs):
    # ---- train ----
    model.train(); run_loss = 0.0; n = 0
    for Pb, Cb in train_loader:
        Pb, Cb = Pb.to(device, non_blocking=True), Cb.to(device, non_blocking=True)
        optimizer.zero_grad()
        curves_pred = model(Pb)
        loss = weighted_curve_loss(curves_pred, Cb)
        loss.backward(); optimizer.step()
        run_loss += loss.item() * Pb.size(0); n += Pb.size(0)
    train_loss = run_loss / n

    # ---- val (loss pondérée + MAE brute non pondérée pour comparaison) ----
    model.eval(); run_val = 0.0; nv = 0; run_mae = 0.0
    with torch.no_grad():
        for Pb, Cb in val_loader:
            Pb, Cb = Pb.to(device, non_blocking=True), Cb.to(device, non_blocking=True)
            curves_pred = model(Pb)
            run_val += weighted_curve_loss(curves_pred, Cb).item() * Pb.size(0)
            run_mae += torch.mean(torch.abs(curves_pred - Cb)).item() * Pb.size(0)
            nv += Pb.size(0)
    val_loss = run_val / nv
    val_mae = run_mae / nv
    scheduler.step(val_loss)

    print(f"Epoch [{epoch+1:03d}/{num_epochs}] LR {optimizer.param_groups[0]['lr']:.2e} "
          f"Train {train_loss:.6f} Val {val_loss:.6f} ValMAE(courbes brute) {val_mae:.6f}")

    if val_loss < best_val:
        best_val, patience_counter = val_loss, 0
        # ---- sauvegarde atomique (évite le verrou Windows code 32) ----
        tmp_path = best_model_path + ".tmp"
        torch.save(model.state_dict(), tmp_path)
        if os.path.exists(best_model_path):
            os.remove(best_model_path)
        os.rename(tmp_path, best_model_path)
        print("Best surrogate saved.")
    else:
        patience_counter += 1
    if patience_counter >= patience:
        print("Early stopping."); break

print("Best val loss (surrogate):", best_val)
print("ATTENTION : vérifiez R² et MAE par canal avec test_surrogate.py avant la phase 2.")

# Test

In [ ]:
# test_surrogate.py
import os, numpy as np, torch
import matplotlib.pyplot as plt
from torch.utils.data import TensorDataset, DataLoader

batch_size = 128
best_model_path = os.path.join(save_dir, "surrogate_best.pth")

# ---- données test : entrée = params (Y_norm), cible = courbes (X_flat) ----
P_te = np.load(os.path.join(save_dir, "Y_test_norm.npy"))    
C_te = np.load(os.path.join(save_dir, "X_test_flat.npy"))     
print("Params test:", P_te.shape, "Courbes test:", C_te.shape)

P_te_t = torch.tensor(P_te, dtype=torch.float32)
C_te_t = torch.tensor(C_te, dtype=torch.float32)
test_loader = DataLoader(TensorDataset(P_te_t, C_te_t),
                         batch_size=batch_size, shuffle=False,
                         pin_memory=torch.cuda.is_available())

# ---- modèle ----
model = MosfetSurrogate().to(device)
model.load_state_dict(torch.load(best_model_path, map_location=device))
model.eval()
print("Surrogate chargé.")

# ---- inférence ----
all_pred = []
with torch.no_grad():
    for Pb, _ in test_loader:
        Pb = Pb.to(device, non_blocking=True)
        all_pred.append(model(Pb).cpu().numpy())
C_pred = np.vstack(all_pred)       
print("Courbes prédites:", C_pred.shape)

# ============================================================
# 1) MÉTRIQUES GLOBALES (espace normalisé)
# ============================================================
mae_global  = np.mean(np.abs(C_pred - C_te))
rmse_global = np.sqrt(np.mean((C_pred - C_te) ** 2))
# R² global
ss_res = np.sum((C_te - C_pred) ** 2)
ss_tot = np.sum((C_te - np.mean(C_te)) ** 2)
r2_global = 1.0 - ss_res / ss_tot

print("\n===== MÉTRIQUES GLOBALES (espace normalisé) =====")
print(f"MAE  global : {mae_global:.6e}")
print(f"RMSE global : {rmse_global:.6e}")
print(f"R²   global : {r2_global:.6f}")

# ============================================================
# 2) MÉTRIQUES PAR CANAL
# ============================================================
C_pred_3d = C_pred.reshape(-1, N_CHANNELS, N_POINTS)
C_te_3d   = C_te.reshape(-1, N_CHANNELS, N_POINTS)

mae_per_ch  = np.mean(np.abs(C_pred_3d - C_te_3d), axis=(0, 2))   
rmse_per_ch = np.sqrt(np.mean((C_pred_3d - C_te_3d) ** 2, axis=(0, 2)))

print("\n===== MAE / RMSE PAR CANAL (normalisé) =====")
for i, name in enumerate(channel_names):
    print(f"  [{i:2d}] {name:>22s} | MAE = {mae_per_ch[i]:.5e} | RMSE = {rmse_per_ch[i]:.5e}")

# Canaux les plus mal reproduits
worst = np.argsort(mae_per_ch)[::-1][:5]
print("\n5 canaux les MOINS bien reconstruits :")
for i in worst:
    print(f"  [{i:2d}] {channel_names[i]} | MAE = {mae_per_ch[i]:.5e}")

# ============================================================
# DIAGNOSTIC VISUEL : 5 courbes par échantillon
# ============================================================
sample_idx    = [0, 1, 2, 3]
channels_show = [0, 4, 8, 10, 11]

n_ch = len(channels_show)            
ncols = 3                           
nrows = int(np.ceil(n_ch / ncols))  

for s in sample_idx:
    plt.figure(figsize=(14, 7))
    for k, ch in enumerate(channels_show):
        plt.subplot(nrows, ncols, k + 1)
        plt.plot(C_te_3d[s, ch, :],   "b-",  label="Vrai")
        plt.plot(C_pred_3d[s, ch, :], "r--", label="Surrogate")
        plt.title(f"Éch. {s} - canal {ch}\n{channel_names[ch]}", fontsize=9)
        plt.xlabel("Point Vg"); plt.ylabel("valeur normalisée")
        plt.grid(True); plt.legend(fontsize=8)
    plt.suptitle(f"Échantillon {s}", fontsize=11)
    plt.tight_layout()
    plt.show()
    
# ============================================================
# 4) HISTOGRAMME DES ERREURS
# ============================================================
plt.figure(figsize=(7, 4))
plt.hist((C_pred - C_te).ravel(), bins=100)
plt.title("Distribution des erreurs surrogate (normalisé)")
plt.xlabel("erreur (pred - vrai)"); plt.ylabel("comptage")
plt.grid(True); plt.tight_layout(); plt.show()

print("\nTest surrogate terminé.")